In [1]:
import torch
import os
import time
from ultralytics import YOLO
from ultralytics.utils.plotting import plot_results

data_yaml = "/home_data/hejx/pre_dataset/driving_data.yaml"
train_output_dir = "/home_data/hejx/runs/detect/train_safety_optimized"
model_size = "s"
model_name = f"yolov8{model_size}.pt"

epochs = 80
batch_size = 16
imgsz = 640
patience = 20
device = 0 if torch.cuda.is_available() else "cpu"

def optimize_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"使用GPU训练：{gpu_name}（总内存：{gpu_mem:.2f} GB）")
    else:
        print("未检测到GPU，将使用CPU训练（速度极慢！）")

optimize_gpu()

def check_dataset():
    if not os.path.exists(data_yaml):
        raise FileNotFoundError(f"数据集配置文件不存在：{data_yaml}")
    train_img_path = os.path.join("/home_data/hejx/pre_dataset", "images/train")
    val_img_path = os.path.join("/home_data/hejx/pre_dataset", "images/val")
    if not os.path.exists(train_img_path) or len(os.listdir(train_img_path)) == 0:
        raise FileNotFoundError(f"训练集图像为空：{train_img_path}")
    if not os.path.exists(val_img_path) or len(os.listdir(val_img_path)) == 0:
        raise FileNotFoundError(f"验证集图像为空：{val_img_path}")
    print("数据集检查通过！")

check_dataset()

print("\n开始训练优化版YOLOv8模型（安全驾驶监测）...")
start_time = time.time()

model = YOLO(model_name)

results = model.train(
    data=data_yaml,
    epochs=epochs,
    batch=batch_size,  # 训练时统一用batch（和验证保持一致）
    imgsz=imgsz,
    device=device,
    patience=patience,
    save=True,
    save_period=10,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0001,
    warmup_epochs=5,
    box=7.5,
    cls=1.5,
    dfl=1.5,
    augment=True,
    perspective=0.0,
    hsv_h=0.05,
    hsv_s=0.8,
    hsv_v=0.5,
    fliplr=0.7,
    mosaic=0.9,
    mixup=0.3,
    copy_paste=0.2,
    dropout=0.05,
    name="yolov8_driving_safety_optimized",
    exist_ok=True,
    verbose=True,
    seed=42,
    project=train_output_dir,
    cos_lr=True,
)

end_time = time.time()
train_hours = (end_time - start_time) / 3600

print("\n" + "="*60)
print("🎉 优化版训练完成！")
print(f"总训练时间：{train_hours:.2f} 小时")
print(f"最优模型保存路径：{results.save_dir}/weights/best.pt")
print(f"最后一轮模型保存路径：{results.save_dir}/weights/last.pt")
print("="*60)

# 修复plot_results调用
plot_results(os.path.join(results.save_dir, "results.csv"))
print(f"\n📊 训练结果曲线图已保存至：{results.save_dir}/results.png")

print("\n📈 开始验证最优模型...")
best_model = YOLO(f"{results.save_dir}/weights/best.pt")
val_metrics = best_model.val(
    data=data_yaml,
    device=device,
    batch=batch_size,  # 关键修复：batch_size → batch
    imgsz=imgsz,
    save_json=True,
    verbose=True
)
print(f"验证集整体mAP50：{val_metrics.box.map50:.4f}")
print(f"验证集整体mAP50-95：{val_metrics.box.map:.4f}")

for i, c in enumerate(val_metrics.names):
    print(f"类别 {c}：P={val_metrics.box.p[i]:.4f}, R={val_metrics.box.r[i]:.4f}, mAP50={val_metrics.box.ap50[i]:.4f}")

✅ 使用GPU训练：Tesla T4（总内存：15.66 GB）
✅ 数据集检查通过！

🚀 开始训练优化版YOLOv8模型（安全驾驶监测）...
New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.245 🚀 Python-3.11.13 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14931MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home_data/hejx/pre_dataset/driving_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.7, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.05, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.3, mode=train, model=yolov8s.pt, mo